In [1]:
import os
import re
import scanpy as sc
import pandas as pd
import numpy as np

In [2]:
# ==============================================================================
# 10x Visium Data Preparation for NMF and Correlation Analysis
# Customized for specific Sample and Organ lists
# ==============================================================================
# Data Directory
samples_base_dir = '/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Samples'
estimate_base_dir = '/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/processed/tumor_purity_estimate'

# Defined Lists
Primary_list = ["PT-1A", "PT-2A", "PT-3A", "PT-3_PRI2", "PT-4A", "PT-5A", "PT-6A", "PT-6_PRI2", "PT-6_PRI3", "PT-7A", "PT-8A", "PT-8_PRI2", "PT-9A", "PT-10A", "PT-11A", "PT-12A", "PT-13A", "PT-13_PRI2", "PT-13_PRI3"]
Lung_list = ["PT-2B", "PT-4C", "PT-10C", "PT-11C", "PT-12C", "PT-13C"]
Liver_list = ["PT-3B", "PT-4B", "PT-5B", "PT-5C", "PT-6B", "PT-7B", "PT-7C", "PT-8B", "PT-8C", "PT-9B", "PT-9C", "PT-10B", "PT-11B", "PT-12B", "PT-13B"]
Peritoneal_list = ["PT-1C", "PT-2C", "PT-3C", "PT-4D", "PT-6C", "PT-6D", "PT-7D", "PT-8D", "PT-9D", "PT-10D", "PT-11D", "PT-12D", "PT-13D", "PT-13E"]

In [3]:
# Create a mapping dictionary for O(1) lookup
organ_mapping = {}
for s in Primary_list: organ_mapping[s] = 'Primary'
for s in Lung_list: organ_mapping[s] = 'Lung'
for s in Liver_list: organ_mapping[s] = 'Liver'
for s in Peritoneal_list: organ_mapping[s] = 'Peritoneal'

In [4]:
all_spots_info = []
all_adata = []

In [5]:
for sample_name in os.listdir(samples_base_dir):
    
    # [1] check if the sample is on the list, if no then skip
    if sample_name not in organ_mapping:
        continue 
        
    # [2] define sample's h5 and gct path
    sample_path = os.path.join(samples_base_dir, sample_name)
    h5_path = os.path.join(sample_path, 'filtered_feature_bc_matrix.h5')
    gct_path = os.path.join(estimate_base_dir, sample_name, f"{sample_name}_estimate_scores.gct")
    
    # dows h5 exist
    if not os.path.exists(h5_path):
        print(f"Skip {sample_name}：can't locate filtered_feature_bc_matrix.h5")
        continue
        
    print(f"正在处理样本: {sample_name} ...")
    
    # [3] extract patient #
    match = re.match(r'(PT-\d+)', sample_name)
    patient_id = match.group(1) if match else sample_name
    
    # [4] extract organ
    organ_name = organ_mapping[sample_name]
    
    # [5] read 10x visium data
    adata = sc.read_10x_h5(h5_path)
    adata.var_names_make_unique()
    
    # [6] spotID
    adata.obs.index = adata.obs.index + '_' + sample_name
    
    # [7] count the reads
    adata.obs['Reads'] = adata.X.sum(axis=1)
    
    # [8] skip MetScore because Singscore is a R package
        
    # [9] read ESTIMATE files
    est_tumor_purity = []
    est_immune = []
    est_stroma = []
    
    if os.path.exists(gct_path):
        try:
            with open(gct_path, 'r') as f:
                gct_lines = f.readlines()
            
            
            gct_header = gct_lines[2].strip().split('\t')
            gct_spots = [s.replace('.', '-') for s in gct_header[2:]]
            
            gct_scores = {}
            for line in gct_lines[3:]:
                parts = line.strip().split('\t')
                score_name = parts[0]
                
                values = [float(v) for v in parts[2:]]
                gct_scores[score_name] = dict(zip(gct_spots, values))
            
            immune_dict = gct_scores.get('ImmuneScore', {})
            stroma_dict = gct_scores.get('StromalScore', {})
            estimate_dict = gct_scores.get('ESTIMATEScore', {})
            
            
            for spot_id in adata.obs.index:
                raw_barcode = spot_id.split('_')[0] 
                
                i_score = immune_dict.get(raw_barcode, np.nan)
                s_score = stroma_dict.get(raw_barcode, np.nan)
                e_score = estimate_dict.get(raw_barcode, np.nan)
                
                
                if not np.isnan(e_score):
                    purity = np.cos(0.6049872018 + 0.0001467884 * e_score)
                else:
                    purity = np.nan
                    
                est_immune.append(i_score)
                est_stroma.append(s_score)
                est_tumor_purity.append(purity)
                
        except Exception as e:
            print(f"⚠️ 解析 {sample_name} 的 GCT 文件时出错: {e}")
            
            est_tumor_purity = [np.nan] * len(adata)
            est_immune = [np.nan] * len(adata)
            est_stroma = [np.nan] * len(adata)
    else:
        
        est_tumor_purity = [np.nan] * len(adata)
        est_immune = [np.nan] * len(adata)
        est_stroma = [np.nan] * len(adata)
            
    # [10] 构建当前切片的 Info 表格，严格对应你要求的 8 个列顺序
    sample_info = pd.DataFrame({
        'Spot_ID': adata.obs.index,
        'Patient_Source': patient_id,              # 第一列：样本病人来源
        'Sample_Name': sample_name,                # 第二列：样本名称
        'Organ': organ_name,                       # 第三列：取样器官
        'Reads': adata.obs['Reads'],               # 第四列：reads
        'MET_Score': 0,                            # 第五列：metscore
        'ESTIMATE_TumorPurity': est_tumor_purity,  # 第六列：tumor purity
        'ESTIMATE_ImmuneScore': est_immune,        # 第七列：immune score
        'ESTIMATE_StromaScore': est_stroma         # 第八列：stroma score
    })
    
    all_spots_info.append(sample_info)
    all_adata.append(adata)

print("---------- 所有样本遍历与解析完成 ----------\n")

正在处理样本: PT-3_PRI2 ...
正在处理样本: PT-6A ...


/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


正在处理样本: PT-12D ...


/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


正在处理样本: PT-4C ...


...

/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


正在处理样本: PT-13B ...


/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


正在处理样本: PT-5A ...
---------- 所有样本遍历与解析完成 ----------



/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/users/hhuan40/.local/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [ ]:
# =====================================================================
# Info chart
# =====================================================================
final_info_df = pd.concat(all_spots_info, ignore_index=True)
final_info_df.to_csv('/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/processed/Visium_AllSpots_InfoTable_noMS.csv', index=False)
print("✅ Info 大表已成功保存为: Visium_AllSpots_InfoTable_Final.csv")


# =====================================================================
# Expression chart
# =====================================================================
print("正在合并基因表达矩阵用于 NMF (行 = 基因, 列 = Spot)...")

if len(all_adata) > 1:
    combined_adata = all_adata[0].concatenate(all_adata[1:], join='inner', batch_key=None)
else:
    combined_adata = all_adata[0]

# 如果内存吃紧，可取消下面两行的注释来只保留高变基因
# sc.pp.highly_variable_genes(combined_adata, n_top_genes=3000)
# combined_adata = combined_adata[:, combined_adata.var['highly_variable']]

if hasattr(combined_adata.X, 'toarray'):
    expr_matrix = combined_adata.X.toarray()
else:
    expr_matrix = combined_adata.X

# 转置矩阵：使行变成基因，列变成 Spot
expr_df = pd.DataFrame(
    expr_matrix,
    index=combined_adata.obs.index,
    columns=combined_adata.var_names
).T

expr_df.index.name = 'Gene'
expr_df.to_csv('/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/processed/Visium_Expression_Matrix_for_NMF.csv')
print("✅ NMF 基因表达矩阵已成功保存为: Visium_Expression_Matrix_for_NMF.csv")

✅ Info 大表已成功保存为: Visium_AllSpots_InfoTable_Final.csv
正在合并基因表达矩阵用于 NMF (行 = 基因, 列 = Spot)...


/tmp/ipykernel_41597/4271213033.py:15: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  combined_adata = all_adata[0].concatenate(all_adata[1:], join='inner', batch_key=None)
